# JS09 — Lab 5: Uji Performansi ANN dengan Recall Query

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membandingkan ANNOY, FAISS IVF, dan HNSW memakai metrik **recall@10** dan **waktu query**, lalu memvisualisasikan *trade-off* akurasi vs kecepatan seperti grafik ANN-Benchmarks.

**Prasyarat:** library `annoy`, `faiss`, dan `hnswlib`. Tidak ada dataset eksternal — 100.000 vektor 128D dibangkitkan secara acak.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Lab 5 — Uji Performansi ANN dengan Recall Query

Modul: `JS09 › Lab 5 — Uji Performansi ANN dengan Recall Query`

Cara standar membandingkan ANN adalah metrik **recall**: berapa banyak tetangga hasil pencarian *approximate* yang benar-benar sama dengan hasil *exact nearest neighbor*. Grafik ANN-Benchmarks memakai sumbu X **recall** dan sumbu Y **queries per second (QPS)**: kanan atas = akurat **dan** cepat.

Karena ANN-Benchmarks memerlukan Docker (tidak tersedia di Colab), kita membuat grafik sederhana dengan sumbu Y **waktu query (ms/query, skala log)** yang dibalik — semakin ke atas berarti semakin cepat, sehingga bentuk kurvanya mirip grafik aslinya.

Konfigurasi: 100.000 vektor **128 dimensi**, 1.000 query, `k = 10`, ground truth dari FAISS brute force (`IndexFlatL2`). Parameter yang dipindai: ANNOY `search_k`, FAISS `nprobe`, HNSW `ef`.

> **Perkiraan waktu di Colab CPU:** ±1–2 menit total (HNSW membangun indeks 4 kali, sekali per nilai `ef`).

In [ ]:
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
import matplotlib.pyplot as plt

# -------------------------------
# Dataset random
# -------------------------------
d = 128        # dimensi
nb = 100000    # jumlah database vector
nq = 1000      # jumlah query

np.random.seed(42)
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')

# -------------------------------
# Ground truth dengan FAISS brute force
# -------------------------------
index_flat = faiss.IndexFlatL2(d)
index_flat.add(xb)
k = 10
_, gt_idx = index_flat.search(xq, k)

# -------------------------------
# Fungsi recall
# -------------------------------
def recall_at_k(I_pred, I_gt, k):
    correct = 0
    for i in range(len(I_pred)):
        correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))
    return correct / (len(I_pred) * k)

# -------------------------------
# Benchmark Annoy
# -------------------------------
def run_annoy(xb, xq, n_trees=10, search_k=1000, k=10):
    f = xb.shape[1]
    index = AnnoyIndex(f, 'euclidean')
    for i, v in enumerate(xb):
        index.add_item(i, v)
    index.build(n_trees)

    start = time.time()
    I = [index.get_nns_by_vector(v, k, search_k=search_k) for v in xq]
    elapsed = (time.time() - start) * 1000 / len(xq)  # ms/query
    rec = recall_at_k(I, gt_idx, k)
    return rec, elapsed

# -------------------------------
# Benchmark FAISS IVF
# -------------------------------
def run_faiss(xb, xq, nlist=100, nprobe=10, k=10):
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
    index.train(xb)
    index.add(xb)

    index.nprobe = nprobe
    start = time.time()
    _, I = index.search(xq, k)
    elapsed = (time.time() - start) * 1000 / len(xq)
    rec = recall_at_k(I, gt_idx, k)
    return rec, elapsed

# -------------------------------
# Benchmark HNSW
# -------------------------------
def run_hnsw(xb, xq, ef=100, M=16, k=10):
    num_elements = xb.shape[0]
    p = hnswlib.Index(space='l2', dim=d)
    p.init_index(max_elements=num_elements, ef_construction=200, M=M)
    p.add_items(xb)
    p.set_ef(ef)

    start = time.time()
    I, _ = p.knn_query(xq, k)
    elapsed = (time.time() - start) * 1000 / len(xq)
    rec = recall_at_k(I, gt_idx, k)
    return rec, elapsed

# -------------------------------
# Jalankan benchmark dengan beberapa parameter
# -------------------------------
results = {"Annoy": [], "Faiss": [], "HNSW": []}

# Annoy
for sk in [200, 500, 1000, 2000]:
    rec, t = run_annoy(xb, xq, n_trees=10, search_k=sk)
    results["Annoy"].append((rec, t))

# FAISS
for npb in [1, 5, 10, 20]:
    rec, t = run_faiss(xb, xq, nlist=100, nprobe=npb)
    results["Faiss"].append((rec, t))

# HNSW
for ef in [50, 100, 200, 400]:
    rec, t = run_hnsw(xb, xq, ef=ef)
    results["HNSW"].append((rec, t))

# -------------------------------
# Visualisasi trade-off
# -------------------------------
plt.figure(figsize=(8,6))
for label, color in zip(results.keys(), ["blue","green","red"]):
    recall, qtime = zip(*results[label])
    plt.plot(recall, qtime, marker="o", label=label, color=color)

plt.xlabel("Recall")
plt.ylabel("Query Time (ms/query, log scale)")
plt.yscale("log")
plt.gca().invert_yaxis()  # invert Y, makin kanan makin turun
plt.title("ANN Benchmark (Annoy vs Faiss vs HNSW)")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.6)
plt.show()

**Catatan kode modul:**

- `run_faiss` dan `run_hnsw` memakai variabel **global** `d` (dimensi). Kalau `d` diganti nama, kedua fungsi ikut rusak.
- Setiap fungsi mengembalikan `elapsed` dalam **ms per query** (total waktu dibagi jumlah query), bukan total detik.
- ANNOY di-build ulang pada setiap nilai `search_k` walaupun pohonnya sama — kode modul apa adanya; waktu build tidak ikut diukur.
- `plt.gca().invert_yaxis()` membuat sumbu Y terbalik: setelah inversi, posisi **atas** = waktu kecil = lebih cepat.

In [ ]:
# [tambahan] Rekap hasil dalam tabel: recall, ms/query, QPS
import pandas as pd

PARAM = {
    'Annoy': ['search_k=200', 'search_k=500', 'search_k=1000', 'search_k=2000'],
    'Faiss': ['nprobe=1', 'nprobe=5', 'nprobe=10', 'nprobe=20'],
    'HNSW':  ['ef=50', 'ef=100', 'ef=200', 'ef=400'],
}

rows = []
for nama, nilai in results.items():
    for parameter, (rec, ms) in zip(PARAM[nama], nilai):
        rows.append({'algoritma': nama,
                     'parameter': parameter,
                     'recall@10': rec,
                     'ms/query': ms,
                     'QPS': 1000 / ms if ms else float('nan')})

rekap_lab5 = pd.DataFrame(rows).round(4)
rekap_lab5

### [tambahan] Grafik recall vs QPS

Versi yang lebih dekat ke ANN-Benchmarks: sumbu X recall, sumbu Y QPS (bukan ms). Titik **kanan atas** adalah yang terbaik.

In [ ]:
# [tambahan] Grafik recall vs QPS
plt.figure(figsize=(8, 6))
for label, color in zip(rekap_lab5['algoritma'].unique(), ['blue', 'green', 'red']):
    sub = rekap_lab5[rekap_lab5['algoritma'] == label]
    plt.plot(sub['recall@10'], sub['QPS'], marker='o', label=label, color=color)

plt.xlabel('Recall@10')
plt.ylabel('QPS (query per detik)')
plt.title('Trade-off recall vs QPS (100k vektor, 128D)')
plt.legend()
plt.grid(True, ls='--', alpha=0.6)
plt.show()

**Analisis Lab 5:** TBD

*(Jelaskan: (1) algoritma mana yang paling cepat pada recall tinggi, (2) bagaimana kenaikan parameter memengaruhi recall dan QPS untuk masing-masing library, (3) mengapa FAISS IVF dengan `nprobe` kecil bisa jauh lebih cepat tetapi recall-nya rendah, (4) interpretasi posisi kanan-atas pada grafik.)*

---
## Kesimpulan Lab 5

**Nilai recall@10 tiap algoritma pada parameter terbaiknya:** TBD

**Urutan kecepatan (QPS) pada recall yang sebanding:** TBD

**Algoritma terbaik untuk kasus 100k vektor 128D menurut percobaan ini:** TBD

**Kesimpulan umum tentang trade-off akurasi vs kecepatan:** TBD